In [4]:
import { PGlite } from "npm:@electric-sql/pglite";
import { pg_trgm } from "npm:@electric-sql/pglite/contrib/pg_trgm";
import { unaccent } from "npm:@electric-sql/pglite/contrib/unaccent";

import { pgDump } from "npm:@electric-sql/pglite-tools/pg_dump";

const client = new PGlite(
  process.env.test ? "memory://" : "./../../web/.data/database",
  {
    extensions: { pg_trgm, unaccent },
  },
);

const dump = await pgDump({ pg: client });
let dumpContent = await dump.text();

dumpContent = dumpContent
  .split("\n")
  .filter((line) => !line.includes("OWNER TO postgres"))
  .join("\n");

// 2. Fix search path resolution for unaccent dictionary
dumpContent = dumpContent.replace(
  "'unaccent'::regdictionary",
  "'public.unaccent'::regdictionary",
);

// Write the dump to a file
await Deno.writeTextFile("./../data/dump.sql", dumpContent);
